# 08 — Access control

**Scenario.** Hybrid search ranks the Restricted break-glass PDF and the MNPI watch book. If those chunks enter the prompt, the model can leak them. **ACL runs on retrieval**, not as a request to the model.

Rules in this demo:

- Restricted → security and executive only
- MNPI → compliance and executive only (ethical wall; security does not see it)

1. Search for cubbies and for WB-8841 with no ACL.
2. Filter the same hits by role.

Context:

- Access control belongs before prompt construction.
- The model cannot leak a chunk it never receives.
- MNPI and Restricted are different walls: security can see Restricted, but not MNPI.

Edge cases:

- A high similarity score does not override need-to-know.
- Mixed prompts can create side-channel confirmation ("yes, that deal exists").
- Public-side staff should receive abstention, not a sanitized confirmation.


Definitions and context:

| Concept | Meaning |
| --- | --- |
| ACL | Access-control list or rule that decides who can read a chunk |
| Need-to-know | Authorization based on role and classification |
| Restricted | High-risk operational security material |
| MNPI | Material nonpublic information behind an ethical wall |
| Side channel | A leak through confirmation, denial, timing, or partial detail |

Do not ask the model to "be careful" with secrets. Remove forbidden chunks
before the prompt exists. This keeps security deterministic and auditable.


In [ ]:
import _course
from lib.pipeline import apply_acl, embed, load_index, rrf

index = load_index()
q = "Who holds break-glass cubbies, and what is WB-8841?"
vec = embed([q])[0]
fused = rrf([index.dense(vec, 16), index.sparse(q, 16)])
print("unfiltered:", sorted({h.chunk.doc_id for h in fused[:10]}))
print()
for role in ("employee", "advisor", "compliance", "security", "executive"):
    vis = apply_acl(fused, role)
    print(
        f"{role:12} restricted={sum(h.chunk.classification=='restricted' for h in vis)}  "
        f"mnpi={sum(h.chunk.classification=='mnpi' for h in vis)}"
    )


The model never sees dropped chunks. Next: `09-pii-and-injection.ipynb`.
